# Fine-Tuning on AMD (ROCm PyTorch, Windows, 20 GB) -> GGUF for LM Studio

Plain PyTorch + Hugging Face `transformers` / `peft`. No Unsloth, no bitsandbytes (both need NVIDIA), and no `trl` (it needs `torch.distributed`, missing from Windows ROCm builds).
LoRA runs in bf16, so VRAM is the limit: 1-3B is comfortable on 20 GB, 7B is tight (needs batch 1, short sequences).

## 1. Install extra libraries (do NOT reinstall torch)

In [ ]:
%pip install -U transformers peft datasets accelerate gguf sentencepiece protobuf

## 2. Check the GPU is visible

In [ ]:
import torch
print(torch.__version__, "| GPU available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "ROCm PyTorch can't see the GPU - fix this before continuing"
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), "GB")
print("bf16 supported:", torch.cuda.is_bf16_supported())

## 3. Configuration

Use the **Hugging Face (safetensors) version** of the model, not the GGUF from LM Studio. Pick the same family you use in LM Studio:

- `Qwen/Qwen2.5-3B-Instruct` (default, ungated)
- `Qwen/Qwen2.5-1.5B-Instruct` (fast)
- `Qwen/Qwen2.5-7B-Instruct` (tight on 20 GB: set BATCH_SIZE=1, MAX_SEQ_LEN=1024)
- `meta-llama/Llama-3.2-3B-Instruct` (gated: accept the licence on HF, run `huggingface-cli login`)
- `google/gemma-2-2b-it` (gated), `microsoft/Phi-3.5-mini-instruct`

In [ ]:
BASE_MODEL    = "Qwen/Qwen2.5-3B-Instruct"
OUTPUT_NAME   = "my-finetuned-model"
DATA_FILE     = "train.jsonl"
MAX_SEQ_LEN   = 1024

LORA_RANK     = 16
EPOCHS        = 3
LEARNING_RATE = 2e-4
BATCH_SIZE    = 2
GRAD_ACCUM    = 4
GGUF_TYPE     = "q8_0"   # q8_0 (near-lossless), f16 (biggest), bf16. See step 11 for q4_k_m.

## 4. Data

`train.jsonl`, one chat per line:
```json
{"messages": [{"role": "user", "content": "question"}, {"role": "assistant", "content": "ideal answer"}]}
```
If the file is missing, a tiny sample is written so you can test the pipeline. Replace it with your real data (100-500+ clean, varied examples).

In [ ]:
import json, os
from datasets import load_dataset

if not os.path.exists(DATA_FILE):
    print("No train.jsonl - writing a SAMPLE for testing.")
    sample = [
        ("What's your name?", "I'm Sparky, a friendly assistant who keeps answers short and clear."),
        ("Explain what an API is.", "An API is a menu of requests one program can make to another. You ask in a set format and get a predictable answer."),
        ("Give me a tip for studying.", "Study in 25-minute bursts, then take a 5-minute break. Test yourself instead of rereading."),
        ("What is a variable in programming?", "A variable is a named box that holds a value you can read or change later."),
        ("How do I stay focused?", "Silence your phone, pick one task, and set a timer. Start with just 10 minutes."),
    ] * 20
    with open(DATA_FILE, "w", encoding="utf-8") as f:
        for q, a in sample:
            f.write(json.dumps({"messages": [{"role": "user", "content": q},
                                              {"role": "assistant", "content": a}]}) + "\n")

raw = load_dataset("json", data_files=DATA_FILE, split="train")
print(raw)

## 5. Load model and tokenizer (bf16)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    attn_implementation="sdpa",
    device_map={"": 0},
)
model.config.use_cache = False
print("Loaded. VRAM used:", round(torch.cuda.memory_allocated()/1e9, 1), "GB")

## 6. Attach LoRA

In [ ]:
from peft import LoraConfig, get_peft_model

model.gradient_checkpointing_enable()
model.enable_input_require_grads()

lora = LoraConfig(
    r=LORA_RANK, lora_alpha=LORA_RANK * 2, lora_dropout=0.05, bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

## 7. Apply the model's chat template

In [ ]:
def to_text(ex):
    return {"text": tokenizer.apply_chat_template(ex["messages"], tokenize=False, add_generation_prompt=False)}

dataset = raw.map(to_text, remove_columns=raw.column_names)
if len(dataset) >= 50:
    s = dataset.train_test_split(test_size=0.1, seed=42)
    train_ds, eval_ds = s["train"], s["test"]
else:
    train_ds, eval_ds = dataset, None
print(train_ds[0]["text"])

def tokenize(ex):
    return tokenizer(ex["text"], truncation=True, max_length=MAX_SEQ_LEN)

train_tok = train_ds.map(tokenize, remove_columns=["text"])
eval_tok  = eval_ds.map(tokenize, remove_columns=["text"]) if eval_ds is not None else None

## 8. Train

In [ ]:
import math, time
from torch.utils.data import DataLoader

# Plain PyTorch loop: Trainer/trl import torch.distributed, which Windows ROCm builds lack.
def collate(batch):
    L = max(len(b["input_ids"]) for b in batch)
    ids = torch.full((len(batch), L), tokenizer.pad_token_id, dtype=torch.long)
    att = torch.zeros((len(batch), L), dtype=torch.long)
    lab = torch.full((len(batch), L), -100, dtype=torch.long)
    for i, b in enumerate(batch):
        t = torch.tensor(b["input_ids"]); n = len(t)
        ids[i, :n] = t; att[i, :n] = 1; lab[i, :n] = t
    return ids, att, lab

train_loader = DataLoader(train_tok, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
eval_loader  = DataLoader(eval_tok, batch_size=BATCH_SIZE, collate_fn=collate) if eval_tok is not None else None

params = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.AdamW(params, lr=LEARNING_RATE, weight_decay=0.01)

steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM)
total_steps = steps_per_epoch * EPOCHS
warmup = max(1, int(0.05 * total_steps))
def lr_lambda(s):
    if s < warmup:
        return (s + 1) / warmup
    return 0.5 * (1 + math.cos(math.pi * (s - warmup) / max(1, total_steps - warmup)))
sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)

def run_batch(ids, att, lab):
    ids, att, lab = ids.cuda(), att.cuda(), lab.cuda()
    with torch.autocast("cuda", dtype=torch.bfloat16):
        return model(input_ids=ids, attention_mask=att, labels=lab).loss

step, t0 = 0, time.time()
model.train()
for epoch in range(EPOCHS):
    running, count = 0.0, 0
    for i, batch in enumerate(train_loader):
        loss = run_batch(*batch)
        (loss / GRAD_ACCUM).backward()
        running += loss.item(); count += 1
        if (i + 1) % GRAD_ACCUM == 0 or (i + 1) == len(train_loader):
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            opt.step(); sched.step(); opt.zero_grad(set_to_none=True)
            step += 1
            if step % 5 == 0:
                print(f"epoch {epoch+1} step {step}/{total_steps} loss {running/count:.4f} "
                      f"lr {sched.get_last_lr()[0]:.2e} ({time.time()-t0:.0f}s)")
                running, count = 0.0, 0
    if eval_loader is not None:
        model.eval(); tot, n = 0.0, 0
        with torch.no_grad():
            for batch in eval_loader:
                tot += run_batch(*batch).item(); n += 1
        print(f"== epoch {epoch+1} eval loss {tot/n:.4f}")
        model.train()

print("Peak VRAM:", round(torch.cuda.max_memory_allocated()/1e9, 1), "GB")

If eval loss rises while train loss falls, you're overfitting: fewer epochs, lower learning rate, more varied data. Out of memory: lower `MAX_SEQ_LEN`, set `BATCH_SIZE=1`, lower `LORA_RANK`, or use a smaller model.

## 9. Test it

In [ ]:
model.config.use_cache = True
model.eval()

def ask(prompt, max_new_tokens=200):
    ids = tokenizer.apply_chat_template([{"role": "user", "content": prompt}],
                                        add_generation_prompt=True, return_tensors="pt", return_dict=True).to("cuda")
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=True, temperature=0.7)
    return tokenizer.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)

print(ask("What's your name?"))
print("---")
print(ask("Explain what a database is."))

## 10. Save adapter, then merge into a full model

In [ ]:
model.save_pretrained(f"{OUTPUT_NAME}-lora")
tokenizer.save_pretrained(f"{OUTPUT_NAME}-lora")

merged = model.merge_and_unload()
MERGED_DIR = f"{OUTPUT_NAME}-merged"
merged.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)
print("Merged model saved to", MERGED_DIR)

## 11. Convert to GGUF with llama.cpp

In [ ]:
import subprocess, sys

if not os.path.exists("llama.cpp"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp"], check=True)

OUT_GGUF = f"{OUTPUT_NAME}-{GGUF_TYPE}.gguf"
subprocess.run([sys.executable, "llama.cpp/convert_hf_to_gguf.py", MERGED_DIR,
                "--outfile", OUT_GGUF, "--outtype", GGUF_TYPE], check=True)
print("Created", OUT_GGUF, round(os.path.getsize(OUT_GGUF)/1e9, 2), "GB")

**Want a smaller q4_k_m file?** The convert script only does f16/bf16/q8_0. Download a Windows build from https://github.com/ggml-org/llama.cpp/releases (any `llama-*-bin-win-*.zip`), unzip, then run:

```
llama-quantize.exe my-finetuned-model-f16.gguf my-finetuned-model-q4_k_m.gguf Q4_K_M
```
(set `GGUF_TYPE = "f16"` above first).

## 12. Use it in LM Studio

Copy the `.gguf` into LM Studio's models folder using the `publisher\model\file.gguf` layout, then it shows up under My Models:

In [ ]:
import shutil
dest = os.path.join(os.path.expanduser("~"), ".lmstudio", "models", "my-models", OUTPUT_NAME)
os.makedirs(dest, exist_ok=True)
shutil.copy(OUT_GGUF, dest)
print("Copied to", dest, "- restart LM Studio or open My Models.")

If replies look broken in LM Studio, make sure its prompt template matches the base family (Qwen/ChatML, Llama 3, Gemma, etc.).